# پروژه طبقه‌بندی سگ و گربه با CNN
این نوت‌بوک برای اجرای پروژه در Google Colab یا Jupyter آماده شده است. ابتدا فایل‌های پروژه را در محیط قرار دهید یا مخزن پروژه را Clone کنید. دیتاست باید از Kaggle دریافت شود و در این مخزن قرار نگرفته است.

In [ ]:
# در صورت اجرا در Colab، کتابخانه‌های لازم را نصب کنید.
!pip -q install numpy pandas Pillow matplotlib scikit-learn kagglehub
import torch, sys, os
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## ۱. دریافت دیتاست
Kaggle ممکن است برای دسترسی به دیتاست به حساب کاربری/مجوز نیاز داشته باشد. مسیر دانلودشده را به متغیر `DATA_DIR` اختصاص دهید.

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download('bhavikjikadara/dog-and-cat-classification-dataset')
print('Dataset folder:', DATA_DIR)
print('Sample entries:', os.listdir(DATA_DIR)[:10])

## ۲. آموزش مدل پایه و ارزیابی
نسبت تقسیم داده ۷۰/۱۵/۱۵ است. اندازه تصویر پیش‌فرض ۱۲۸×۱۲۸ و تعداد epoch پیش‌فرض ۱۲ است. برای کارت گرافیک با حافظه محدود، batch size را ۱۶ یا ۸ انتخاب کنید.

In [ ]:
import subprocess
cmd = [sys.executable, 'train.py', '--data-dir', DATA_DIR, '--epochs', '12', '--batch-size', '16', '--image-size', '128']
subprocess.run(cmd, check=True)

## ۳. آزمایش اثر تعداد بلوک‌ها و لایه‌های منظم‌سازی
این مرحله مدل‌های دو و چهار بلوکی را با مدل سه بلوکی مقایسه می‌کند و اثر حذف Batch Normalization و Dropout را جداگانه می‌سنجد. این مرحله از آموزش یک مدل بیشتر زمان می‌برد.

In [ ]:
cmd = [sys.executable, 'train.py', '--data-dir', DATA_DIR, '--epochs', '12', '--batch-size', '16', '--image-size', '128', '--ablation-only']
subprocess.run(cmd, check=True)

## ۴. مشاهده نتایج
نمودار MSE برحسب epoch، ماتریس ابهام و گزارش خودکار را بررسی کنید.

In [ ]:
from IPython.display import display, Image as IPyImage
import pandas as pd
display(pd.read_csv('outputs/main/history.csv').tail())
display(IPyImage('outputs/main/mse_epochs.png'))
display(IPyImage('outputs/main/confusion_matrix.png'))
display(pd.read_csv('outputs/ablations/experiment_summary.csv'))
display(IPyImage('outputs/ablations/architecture_comparison.png'))
print(open('outputs/REPORT.md', encoding='utf-8').read())

## ۵. پیش‌بینی روی تصویر جدید
پس از آموزش می‌توان یک تصویر دلخواه را با `predict.py` طبقه‌بندی کرد. مسیر تصویر را جایگزین کنید.

In [ ]:
# نمونه: subprocess.run([sys.executable, 'predict.py', '/content/my_pet.jpg'], check=True)